# Analyse de l'espérance de vie

Deux jeux de données explorant l'espérance de vie dans le monde :
- **Gapminder** (1952–2007) : country, year, population, continent, lifeExp, gdpPercap
- **Espérance de vie OMS (WHO)** (2000–2015) : 193 pays, 22 indicateurs (mortalité, IMC/BMI, PIB, scolarisation, etc.)

Ce classeur démontre l'importation et l'analyse de fichiers CSV en **Python** et en **R**.

## 1. Configuration : installer les paquets & télécharger les jeux de données

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly installés')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Existe déjà : {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Téléchargé : {name}: {lines} lignes")

## 2. Gapminder : Exploration avec Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Dimension (Shape) : {gap.shape}")
print(f"Continents : {sorted(gap['continent'].unique())}")
print(f"Plage d'années : {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Espérance de vie au fil du temps par continent
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Espérance de vie par continent (1952–2007)',
              labels={'lifeExp': 'Espérance de vie (années)', 'year': 'Année'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# PIB vs espérance de vie (2007), taille des bulles = population
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='PIB vs espérance de vie (2007)',
                 labels={'gdpPercap': 'PIB par habitant (log)', 'lifeExp': 'Espérance de vie'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder : Exploration avec R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Distribution de l'espérance de vie par continent (boîte à moustaches)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Espérance de vie par continent",
        xlab = "Continent", ylab = "Espérance de vie (années)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# Top 10 des pays par gain d'espérance de vie (1952 vs 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "Top 10 : Gain d'espérance de vie (1952–2007)",
        xlab = "Années gagnées",
        col = "#00CC96", border = NA)

## 4. Espérance de vie OMS : Exploration avec Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Dimension (Shape) : {who.shape}")
print(f"Colonnes : {list(who.columns)}")
print(f"\nValeurs manquantes (top 5) :")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# Pays en développement vs développés : distributions pré-agrégées de l'espérance de vie
# Les coordonnées de barres explicites s'affichent de manière cohérente via la passerelle Plotly du navigateur.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Espérance de vie : En développement vs Développés',
             labels={'Life expectancy': 'Espérance de vie (années)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Scolarisation vs espérance de vie
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Scolarisation vs espérance de vie (2014)',
                 labels={'Life expectancy': 'Espérance de vie (années)',
                         'Schooling': 'Années de scolarisation'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. Espérance de vie OMS : Exploration avec R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nPays :", length(unique(who$Country)))
cat("\nPlage d'années :", range(who$Year))

In [ ]:
# Corrélation : Mortalité adulte vs espérance de vie
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Mortalité adulte vs espérance de vie",
     xlab = "Mortalité adulte (pour 1000)",
     ylab = "Espérance de vie (années)")
legend("topright", legend = c("Développés", "En développement"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Modèle linéaire simple : quels facteurs prédisent l'espérance de vie ?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Principales conclusions

- L'espérance de vie a progressé à l'échelle mondiale, mais d'importants écarts subsistent entre les continents
- Le PIB et la scolarisation sont de puissants prédicteurs positifs de l'espérance de vie
- La mortalité adulte est le prédicteur négatif le plus marqué
- Les pays en développement présentent une variance beaucoup plus large dans leurs résultats